# Google Drive 视频推送到 YouTube Live

这个 notebook 只保留配置和启动逻辑。真正的推流代码放在 GitHub 的 `github_runtime/youtube_live_push_runtime.py`，Colab 运行时会先下载最新版本再执行。

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg


In [ ]:
import os
from getpass import getpass
from google.colab import drive

drive.mount('/content/drive')

#@title 1. GitHub 运行时代码
GITHUB_RAW_BASE_URL = 'https://raw.githubusercontent.com/collinsgraciano/huya-replay-colab-df-youtube-live/master/github_runtime' #@param {type:"string"}
RUNTIME_FILENAME = 'youtube_live_push_runtime.py' #@param {type:"string"}
RUNTIME_MODULE_NAME = 'youtube_live_push_runtime' #@param {type:"string"}
FORCE_RUNTIME_DOWNLOAD = True #@param {type:"boolean"}

#@title 2. 推流输入与目标
INPUT_VIDEO_PATH = '/content/drive/MyDrive/huya_replay_df/outputs/replace_me_df.mp4' #@param {type:"string"}
STREAM_URL = 'rtmps://a.rtmp.youtube.com/live2' #@param {type:"string"}
STREAM_KEY = '' #@param {type:"string"}

#@title 3. 推流编码参数
LOOP_FOREVER = False #@param {type:"boolean"}
TARGET_HEIGHT = 1080 #@param [480, 720, 1080] {type:"raw"}
FPS = 30 #@param [24, 25, 30, 60] {type:"raw"}
VIDEO_BITRATE = '6000k' #@param ['2500k', '4000k', '6000k', '9000k'] {type:"raw"}
AUDIO_BITRATE = '128k' #@param ['96k', '128k', '160k'] {type:"raw"}
PRESET = 'veryfast' #@param ['ultrafast', 'superfast', 'veryfast', 'faster', 'fast', 'medium'] {type:"raw"}
DRY_RUN_ONLY = False #@param {type:"boolean"}

STREAM_KEY = STREAM_KEY.strip() or os.environ.get('YOUTUBE_STREAM_KEY', '').strip() or getpass('Paste YouTube stream key: ').strip()


In [ ]:
import importlib.util
import sys
import time
import urllib.request
from pathlib import Path

runtime_dir = Path('/content/codex_runtime')
runtime_dir.mkdir(parents=True, exist_ok=True)
runtime_url = f"{GITHUB_RAW_BASE_URL.rstrip('/')}/{RUNTIME_FILENAME}"
download_url = runtime_url
if FORCE_RUNTIME_DOWNLOAD:
    download_url = f"{runtime_url}?ts={int(time.time())}"

local_runtime_path = runtime_dir / RUNTIME_FILENAME
print('Downloading runtime from:', download_url)
urllib.request.urlretrieve(download_url, local_runtime_path)
print('Runtime saved to:', local_runtime_path)

spec = importlib.util.spec_from_file_location(RUNTIME_MODULE_NAME, local_runtime_path)
runtime_module = importlib.util.module_from_spec(spec)
sys.modules[RUNTIME_MODULE_NAME] = runtime_module
spec.loader.exec_module(runtime_module)
print('Runtime module loaded:', runtime_module)


In [ ]:
config = {
    'input_video_path': INPUT_VIDEO_PATH,
    'stream_url': STREAM_URL,
    'stream_key': STREAM_KEY,
    'loop_forever': LOOP_FOREVER,
    'target_height': TARGET_HEIGHT,
    'fps': FPS,
    'video_bitrate': VIDEO_BITRATE,
    'audio_bitrate': AUDIO_BITRATE,
    'preset': PRESET,
    'dry_run_only': DRY_RUN_ONLY,
}

result = runtime_module.run_stream(config)
result
